# 06 — DuckDB harmonisation

Unifies MI-ADHD, Children's Commissioner, OpenSAFELY, and Commons Library outputs into one long fact table. Schema preserves raw source dimensions plus coarse normalised versions. Output: `data/processed/adhd_atlas.duckdb` and a flattened parquet for downstream notebooks and the Streamlit app.

**Author:** Noble Chidera Onyema · **Created:** 18 May 2026
© 2026 Noble Chidera Onyema. All Rights Reserved.

In [1]:
"""
06_harmonisation_duckdb.ipynb — fact table across all sources.

Copyright (c) 2026 Noble Chidera Onyema. All Rights Reserved.
"""

from pathlib import Path
from datetime import datetime
import duckdb
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

DUCKDB_PATH = DATA_PROCESSED / "adhd_atlas.duckdb"
if DUCKDB_PATH.exists():
    DUCKDB_PATH.unlink()

con = duckdb.connect(str(DUCKDB_PATH))
print(f"DuckDB version: {duckdb.__version__}")
print(f"DB path:        {DUCKDB_PATH}")

DuckDB version: 1.1.3
DB path:        C:\Users\HP\Projects\adhd-care-equity-tracker\data\processed\adhd_atlas.duckdb


In [2]:
con.execute("""
CREATE TABLE fact_adhd_metrics (
    date_start         DATE NOT NULL,
    date_end           DATE,
    source             TEXT NOT NULL,
    measure_code       TEXT NOT NULL,
    measure_name       TEXT,
    jurisdiction       TEXT DEFAULT 'England',
    age_band_raw       TEXT,
    age_group_coarse   TEXT,
    sex                TEXT,
    ethnicity_raw      TEXT,
    ethnicity_coarse   TEXT,
    value              DOUBLE,
    value_type         TEXT,
    denominator        DOUBLE,
    notes              TEXT,
    ingested_at        TIMESTAMP
);
""")

result = con.execute("DESCRIBE fact_adhd_metrics").fetchdf()
print(result.to_string(index=False))

     column_name column_type null  key   default extra
      date_start        DATE   NO None      None  None
        date_end        DATE  YES None      None  None
          source     VARCHAR   NO None      None  None
    measure_code     VARCHAR   NO None      None  None
    measure_name     VARCHAR  YES None      None  None
    jurisdiction     VARCHAR  YES None 'England'  None
    age_band_raw     VARCHAR  YES None      None  None
age_group_coarse     VARCHAR  YES None      None  None
             sex     VARCHAR  YES None      None  None
   ethnicity_raw     VARCHAR  YES None      None  None
ethnicity_coarse     VARCHAR  YES None      None  None
           value      DOUBLE  YES None      None  None
      value_type     VARCHAR  YES None      None  None
     denominator      DOUBLE  YES None      None  None
           notes     VARCHAR  YES None      None  None
     ingested_at   TIMESTAMP  YES None      None  None


In [3]:
mi = pd.read_parquet(DATA_PROCESSED / "mi_adhd_feb2026.parquet")

AGE_COARSE = {
    "People aged 0 to 4":   "child",
    "People aged 5 to 17":  "child",
    "People aged 18 to 24": "young_adult",
    "People aged 25+":      "adult",
    "People aged Unknown":  "unknown",
}

mi_long = mi.copy()
mi_long["date_start"]       = pd.to_datetime(mi_long["REPORTING_PERIOD_START_DATE"]).dt.date
mi_long["date_end"]         = pd.to_datetime(mi_long["REPORTING_PERIOD_END_DATE"]).dt.date
mi_long["source"]           = "mi_adhd"
mi_long["measure_code"]     = mi_long["INDICATOR_ID"].astype(str)
mi_long["measure_name"]     = mi_long["INDICATOR_ID"].astype(str)
mi_long["jurisdiction"]     = "England"
mi_long["age_band_raw"]     = mi_long.apply(
    lambda r: r["PRIMARY_LEVEL_DESCRIPTION"] if r["BREAKDOWN"] == "Age Group" else None, axis=1
)
mi_long["age_group_coarse"] = mi_long["age_band_raw"].map(AGE_COARSE)
mi_long["sex"]              = mi_long.apply(
    lambda r: str(r["PRIMARY_LEVEL_DESCRIPTION"]).lower() if r["BREAKDOWN"] == "Gender" else None, axis=1
)
mi_long["ethnicity_raw"]    = mi_long.apply(
    lambda r: r["PRIMARY_LEVEL_DESCRIPTION"] if r["BREAKDOWN"] == "Ethnicity" else None, axis=1
)
mi_long["ethnicity_coarse"] = None
mi_long["value"]            = mi_long["VALUE"]
mi_long["value_type"]       = "count"
mi_long["denominator"]      = None
mi_long["notes"]            = None
mi_long["ingested_at"]      = datetime.utcnow()

mi_long = mi_long[[
    "date_start", "date_end", "source", "measure_code", "measure_name", "jurisdiction",
    "age_band_raw", "age_group_coarse", "sex", "ethnicity_raw", "ethnicity_coarse",
    "value", "value_type", "denominator", "notes", "ingested_at"
]]

con.register("mi_long", mi_long)
con.execute("INSERT INTO fact_adhd_metrics SELECT * FROM mi_long")

result = con.execute("""
    SELECT source, COUNT(*) AS rows, MIN(date_start) AS earliest, MAX(date_start) AS latest
    FROM fact_adhd_metrics
    GROUP BY source
""").fetchdf()
print(result.to_string(index=False))

 source  rows   earliest     latest
mi_adhd  8166 2024-12-01 2026-02-01


In [4]:
OS_TABLES = {
    "table_1": ("ADHD_recorded_prevalence",                   "percentage", "ADHD diagnosis prevalence (recorded)"),
    "table_2": ("ADHD_medication_with_ADHD_diagnosis",        "percentage", "Medication prescribing among diagnosed"),
    "table_3": ("ADHD_medication_without_ADHD_diagnosis",     "percentage", "Medication prescribing without ADHD diagnosis"),
    "table_4": ("ADHD_patients_with_medication_prev_6_months","percentage", "6-month rolling prescribing rate among diagnosed"),
}

OS_AGE_COARSE = {
    "0 to 9":      "child",
    "10 to 17":    "child",
    "18 to 24":    "young_adult",
    "25 to 34":    "adult",
    "35 and over": "adult",
    "35 to 44":    "adult",
    "45 to 54":    "adult",
    "55 to 64":    "adult",
    "65 to 74":    "adult",
    "75 and over": "adult",
}

os_frames = []
for key, (code, vtype, name) in OS_TABLES.items():
    df = pd.read_parquet(DATA_PROCESSED / f"opensafely_nov2025_{key}.parquet")
    df["date_start"]       = pd.to_datetime(df["Reporting_Period_Start_Date"], format="%d/%m/%Y").dt.date
    df["date_end"]         = pd.to_datetime(df["Reporting_Period_End_Date"],   format="%d/%m/%Y").dt.date
    df["source"]           = "opensafely"
    df["measure_code"]     = code
    df["measure_name"]     = name
    df["jurisdiction"]     = "England"
    df["age_band_raw"]     = df["Age_Band"]
    df["age_group_coarse"] = df["Age_Band"].map(OS_AGE_COARSE).fillna("unknown")
    df["sex"]              = df["Sex"].str.lower()
    df["ethnicity_raw"]    = None
    df["ethnicity_coarse"] = None
    df["value"]            = df["Percentage"]
    df["value_type"]       = vtype
    df["denominator"]      = df["Denominator"]
    df["notes"]            = None
    df["ingested_at"]      = datetime.utcnow()
    os_frames.append(df[[
        "date_start", "date_end", "source", "measure_code", "measure_name", "jurisdiction",
        "age_band_raw", "age_group_coarse", "sex", "ethnicity_raw", "ethnicity_coarse",
        "value", "value_type", "denominator", "notes", "ingested_at"
    ]])

# Table 5 is annual, different schema
t5 = pd.read_parquet(DATA_PROCESSED / "opensafely_nov2025_table_5.parquet")
t5["date_start"]       = pd.to_datetime(t5["Year_of_medication"].str[:4] + "-04-01").dt.date
t5["date_end"]         = pd.to_datetime((t5["Year_of_medication"].str[:4].astype(int) + 1).astype(str) + "-03-31").dt.date
t5["source"]           = "opensafely"
t5["measure_code"]     = "Median_time_diagnosis_to_medication_weeks"
t5["measure_name"]     = "Median weeks from ADHD diagnosis to first prescription"
t5["jurisdiction"]     = "England"
t5["age_band_raw"]     = t5["Age_Band"]
t5["age_group_coarse"] = t5["Age_Band"].map(OS_AGE_COARSE).fillna("unknown")
t5["sex"]              = t5["Sex"].str.lower()
t5["ethnicity_raw"]    = None
t5["ethnicity_coarse"] = None
t5["value"]            = t5["Median"].astype(float)
t5["value_type"]       = "median_weeks"
t5["denominator"]      = t5["size"].astype(float)
t5["notes"]            = None
t5["ingested_at"]      = datetime.utcnow()
os_frames.append(t5[[
    "date_start", "date_end", "source", "measure_code", "measure_name", "jurisdiction",
    "age_band_raw", "age_group_coarse", "sex", "ethnicity_raw", "ethnicity_coarse",
    "value", "value_type", "denominator", "notes", "ingested_at"
]])

os_combined = pd.concat(os_frames, ignore_index=True)
con.register("os_combined", os_combined)
con.execute("INSERT INTO fact_adhd_metrics SELECT * FROM os_combined")

print(f"OpenSAFELY rows inserted: {len(os_combined):,}")

OpenSAFELY rows inserted: 1,728


In [5]:
CCO_DATE_PROXY = pd.Timestamp("2024-09-30").date()  # report covers data through Sep 2024

cco_eth_pct = pd.DataFrame([
    {"ethnicity_raw": "White",                     "ethnicity_coarse": "white",   "value": 71.0},
    {"ethnicity_raw": "Asian or Asian British",    "ethnicity_coarse": "asian",   "value": 1.4},
    {"ethnicity_raw": "Black or Black British",    "ethnicity_coarse": "black",   "value": 3.5},
    {"ethnicity_raw": "Mixed",                     "ethnicity_coarse": "mixed",   "value": 4.8},
    {"ethnicity_raw": "Other",                     "ethnicity_coarse": "other",   "value": 14.0},
    {"ethnicity_raw": "Not known",                 "ethnicity_coarse": "unknown", "value": 5.3},
])
cco_eth_pct["date_start"]       = CCO_DATE_PROXY
cco_eth_pct["date_end"]         = CCO_DATE_PROXY
cco_eth_pct["source"]           = "cco"
cco_eth_pct["measure_code"]     = "ADHD_referral_share_by_ethnicity"
cco_eth_pct["measure_name"]     = "Share of child ADHD referrals by ethnicity (CCO report p108)"
cco_eth_pct["jurisdiction"]     = "England"
cco_eth_pct["age_band_raw"]     = "children (CSDS)"
cco_eth_pct["age_group_coarse"] = "child"
cco_eth_pct["sex"]              = "all"
cco_eth_pct["value_type"]       = "percentage"
cco_eth_pct["denominator"]      = None
cco_eth_pct["notes"]            = "CCO Oct 2024 report, page 108. Comparator: Census 2021 child ethnic composition."
cco_eth_pct["ingested_at"]      = datetime.utcnow()

cco_eth_pct = cco_eth_pct[[
    "date_start", "date_end", "source", "measure_code", "measure_name", "jurisdiction",
    "age_band_raw", "age_group_coarse", "sex", "ethnicity_raw", "ethnicity_coarse",
    "value", "value_type", "denominator", "notes", "ingested_at"
]]
con.register("cco_eth_pct", cco_eth_pct)
con.execute("INSERT INTO fact_adhd_metrics SELECT * FROM cco_eth_pct")

cco_headlines = pd.read_parquet(DATA_PROCESSED / "cco_nd_oct2024_headlines.parquet")
cco_headlines_long = pd.DataFrame({
    "date_start":       [CCO_DATE_PROXY] * len(cco_headlines),
    "date_end":         [CCO_DATE_PROXY] * len(cco_headlines),
    "source":           ["cco"] * len(cco_headlines),
    "measure_code":     ["CCO_headline_" + str(i+1) for i in range(len(cco_headlines))],
    "measure_name":     cco_headlines["metric"],
    "jurisdiction":     ["England"] * len(cco_headlines),
    "age_band_raw":     ["children"] * len(cco_headlines),
    "age_group_coarse": ["child"] * len(cco_headlines),
    "sex":              ["all"] * len(cco_headlines),
    "ethnicity_raw":    [None] * len(cco_headlines),
    "ethnicity_coarse": [None] * len(cco_headlines),
    "value":            [None] * len(cco_headlines),
    "value_type":       ["text"] * len(cco_headlines),
    "denominator":      [None] * len(cco_headlines),
    "notes":            cco_headlines["value_text"] + " | source section: " + cco_headlines["source_section"],
    "ingested_at":      [datetime.utcnow()] * len(cco_headlines),
})
con.register("cco_headlines_long", cco_headlines_long)
con.execute("INSERT INTO fact_adhd_metrics SELECT * FROM cco_headlines_long")

print(f"CCO rows inserted: {len(cco_eth_pct) + len(cco_headlines_long)}")

CCO rows inserted: 9


In [6]:
cl = pd.read_parquet(DATA_PROCESSED / "commons_library_cbp10551_headlines.parquet")
CL_DATE = pd.Timestamp("2025-12-01").date()

cl_long = pd.DataFrame({
    "date_start":       [CL_DATE] * len(cl),
    "date_end":         [CL_DATE] * len(cl),
    "source":           ["commons_library"] * len(cl),
    "measure_code":     ["CBP10551_" + str(i+1) for i in range(len(cl))],
    "measure_name":     cl["metric"],
    "jurisdiction":     ["England"] * len(cl),
    "age_band_raw":     [None] * len(cl),
    "age_group_coarse": ["all"] * len(cl),
    "sex":              ["all"] * len(cl),
    "ethnicity_raw":    [None] * len(cl),
    "ethnicity_coarse": [None] * len(cl),
    "value":            cl["value"].astype("Float64"),
    "value_type":       ["count"] * len(cl),
    "denominator":      [None] * len(cl),
    "notes":            cl["source"],
    "ingested_at":      [datetime.utcnow()] * len(cl),
})
con.register("cl_long", cl_long)
con.execute("INSERT INTO fact_adhd_metrics SELECT * FROM cl_long")

print(f"Commons Library rows inserted: {len(cl_long)}")

Commons Library rows inserted: 6


In [7]:
result = con.execute("""
    SELECT
        source,
        COUNT(*) AS rows,
        MIN(date_start) AS earliest,
        MAX(date_start) AS latest,
        COUNT(DISTINCT measure_code) AS distinct_measures
    FROM fact_adhd_metrics
    GROUP BY source
    ORDER BY source
""").fetchdf()
print(result.to_string(index=False))

print()
total = con.execute("SELECT COUNT(*) AS total FROM fact_adhd_metrics").fetchone()[0]
print(f"Total rows in fact table: {total:,}")

         source  rows   earliest     latest  distinct_measures
            cco     9 2024-09-30 2024-09-30                  4
commons_library     6 2025-12-01 2025-12-01                  6
        mi_adhd  8166 2024-12-01 2026-02-01                 27
     opensafely  1728 2016-04-01 2025-03-01                  5

Total rows in fact table: 9,909


## Validation queries

Three checks. (1) Female ADHD diagnosis growth via OpenSAFELY against published headline. (2) MI-ADHD open-list total at Dec 2025 against the 562,480 published figure. (3) CCO ethnicity shares match what was extracted from the PDF.

In [8]:
# Validation 1: OpenSAFELY female prevalence endpoints
q1 = con.execute("""
    WITH yr AS (
        SELECT
            date_start,
            sex,
            SUM(value * denominator) / NULLIF(SUM(denominator), 0) AS rate_pct
        FROM fact_adhd_metrics
        WHERE source = 'opensafely'
          AND measure_code = 'ADHD_recorded_prevalence'
          AND sex IN ('male', 'female')
        GROUP BY date_start, sex
    )
    SELECT sex,
           MIN(date_start) AS first_yr,
           MAX(date_start) AS last_yr,
           ROUND(MIN(rate_pct), 3) AS first_rate_pct,
           ROUND(MAX(rate_pct), 3) AS last_rate_pct
    FROM yr
    GROUP BY sex
    ORDER BY sex
""").fetchdf()
print("Q1 — OpenSAFELY prevalence endpoints by sex (re-derived from fact table):")
print(q1.to_string(index=False))

# Validation 2: MI-ADHD open referrals total at Dec 2025
q2 = con.execute("""
    SELECT SUM(value) AS total
    FROM fact_adhd_metrics
    WHERE source = 'mi_adhd'
      AND measure_code = 'ADHD003'
      AND age_band_raw IS NOT NULL
      AND date_start = '2025-12-01'
""").fetchone()[0]
print(f"\nQ2 — MI-ADHD ADHD003 total at 2025-12-01: {q2:,.0f}")
print(f"       Official NHS England figure:       562,480")
print(f"       Match: {abs(q2 - 562480) < 1}")

# Validation 3: CCO ethnicity shares
q3 = con.execute("""
    SELECT ethnicity_raw, value AS share_pct
    FROM fact_adhd_metrics
    WHERE source = 'cco'
      AND measure_code = 'ADHD_referral_share_by_ethnicity'
    ORDER BY value DESC
""").fetchdf()
print("\nQ3 — CCO ADHD child referrals by ethnicity:")
print(q3.to_string(index=False))
print(f"Sum: {q3['share_pct'].sum():.1f}%  (expected: 100% ± small rounding)")

Q1 — OpenSAFELY prevalence endpoints by sex (re-derived from fact table):
   sex   first_yr    last_yr  first_rate_pct  last_rate_pct
female 2016-04-01 2024-04-01           0.160          0.923
  male 2016-04-01 2024-04-01           0.694          1.628

Q2 — MI-ADHD ADHD003 total at 2025-12-01: 562,480
       Official NHS England figure:       562,480
       Match: True

Q3 — CCO ADHD child referrals by ethnicity:
         ethnicity_raw  share_pct
                 White       71.0
                 Other       14.0
             Not known        5.3
                 Mixed        4.8
Black or Black British        3.5
Asian or Asian British        1.4
Sum: 100.0%  (expected: 100% ± small rounding)


In [9]:
FLATTENED = DATA_PROCESSED / "adhd_atlas_fact.parquet"
con.execute(f"""
    COPY (SELECT * FROM fact_adhd_metrics)
    TO '{FLATTENED.as_posix()}'
    (FORMAT PARQUET, COMPRESSION 'snappy')
""")
print(f"Saved: {FLATTENED.name}  ({FLATTENED.stat().st_size / 1024:.1f} KB)")
print(f"DuckDB file: {DUCKDB_PATH.name}  ({DUCKDB_PATH.stat().st_size / 1024:.1f} KB)")

Saved: adhd_atlas_fact.parquet  (72.5 KB)
DuckDB file: adhd_atlas.duckdb  (12.0 KB)


## Notebook 06 summary

Saved:
- `data/processed/adhd_atlas.duckdb` — DuckDB file with `fact_adhd_metrics`.
- `data/processed/adhd_atlas_fact.parquet` — flattened version, 9,909 rows.

Four sources, 42 distinct measures, span Apr 2016 – Feb 2026. Schema preserves each source's raw category names plus a coarse normalised view.